In [1]:
#which python
import sys
print(sys.executable)

/Users/mathewsair/Documents/dockerTests/aihero/project/.venv/bin/python


In [2]:
import io
import zipfile
import requests
import frontmatter

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [4]:
openclaw_data = read_repo_data('openclaw', 'openclaw')
print(f"OpenClaw documents: {len(openclaw_data)}")

OpenClaw documents: 868


In [6]:
#print(f"openclaw_data_top: {openclaw_data[4:5]}")

In [9]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result


In [10]:
openclaw_chunks_slidingWindow = []

for doc in openclaw_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    openclaw_chunks_slidingWindow.extend(chunks)


In [11]:
print(len(openclaw_chunks_slidingWindow))


5681


In [12]:
print(openclaw_chunks_slidingWindow[2000])

{'start': 2000, 'chunk': ' host-local).\n\n```\nSystem: [2026-01-12 12:19:17 PST] Model switched.\n```\n\n### Configure user timezone + format\n\n```json5\n{\n  agents: {\n    defaults: {\n      userTimezone: "America/Chicago",\n      timeFormat: "auto", // auto | 12 | 24\n    },\n  },\n}\n```\n\n- `userTimezone` sets the **user-local timezone** for prompt context.\n- `timeFormat` controls **12h/24h display** in the prompt. `auto` follows OS prefs.\n\n## Time format detection (auto)\n\nWhen `timeFormat: "auto"`, OpenClaw inspects the OS preference (macOS/Windows)\nand falls back to locale formatting. The detected value is **cached per process**\nto avoid repeated system calls.\n\n## Tool payloads + connectors (raw provider time + normalized fields)\n\nChannel tools return **provider-native timestamps** and add normalized fields for consistency:\n\n- `timestampMs`: epoch milliseconds (UTC)\n- `timestampUtc`: ISO 8601 UTC string\n\nRaw provider fields are preserved so nothing is lost.\n\

In [25]:
all_paragraphs = []
for item in openclaw_data:
    text = item['content']
    paragraphs = re.split(r"\n\s*\n", text.strip())
    all_paragraphs.extend(paragraphs)

In [29]:
openclaw_chunks_ParaNSlidingWindow = []
for paragraph in all_paragraphs:
    chunks = sliding_window(paragraph, 2000, 1000)
    openclaw_chunks_ParaNSlidingWindow.extend(chunks)

In [43]:
print(openclaw_chunks_ParaNSlidingWindow[0:5])

[{'start': 0, 'chunk': '# OpenClaw Upstream Sync Workflow'}, {'start': 0, 'chunk': 'Use this workflow when your fork has diverged from upstream (e.g., "18 commits ahead, 29 commits behind").'}, {'start': 0, 'chunk': '## Quick Reference'}, {'start': 0, 'chunk': '```bash\n# Check divergence status\ngit fetch upstream && git rev-list --left-right --count main...upstream/main'}, {'start': 0, 'chunk': '# Full sync (rebase preferred)\ngit fetch upstream && git rebase upstream/main && pnpm install && pnpm build && ./scripts/restart-mac.sh'}]


In [33]:
## Chunking by sections
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections

In [34]:
openclaw_chunks_section = []

for doc in openclaw_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        openclaw_chunks_section.append(section_doc)

In [35]:
print(len(openclaw_chunks_section))

5789


In [47]:
print(openclaw_chunks_section[4])

{'description': 'Update OpenClaw from upstream when branch has diverged (ahead/behind)', 'filename': 'openclaw-main/.agent/workflows/update_clawdbot.md', 'section': '## Step 3: Rebuild Everything\n\nAfter sync completes:\n\n```bash\n# Install dependencies (regenerates lock if needed)\npnpm install\n\n# Build TypeScript\npnpm build\n\n# Build UI assets\npnpm ui:build\n\n# Run diagnostics\npnpm clawdbot doctor\n```\n\n---'}


In [13]:
openclaw_chunks= openclaw_chunks_slidingWindow

In [14]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [31]:
# New version of text search implemented for agentic tool calling
#def text_search(query):
#    return openclaw_index.search(query, num_results=5)

#def vector_search(query):
#    q = embedding_model.encode(query)
#    return openclaw_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [44]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the Openclaw index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the Openclaw index.
    """
    return openclaw_index.search(query, num_results=5)

def vector_search(query: str) -> List[Any]:
    """
    Perform a semantic search on the Openclaw index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the Openclaw index.
    """
    q = embedding_model.encode(query)
    return openclaw_vindex.search(q, num_results=5)


In [36]:
from minsearch import Index
openclaw_index = Index(
    text_fields=["question", "content"],
    keyword_fields=[]
)

openclaw_index.fit(openclaw_data)


In [37]:
#query = "What is OpenClaw"
query = "Give me an overview of OpenClaw"
text_search_results = text_search(query)

In [38]:
print(text_search_results[1].keys())


dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])


In [39]:
for r in text_search_results:
    print(f"{r.keys()}")

print(f"\n")

#for r in text_search_results:
#    print(f'{r['summary']}')

for r in text_search_results:
    print(f'{r['filename']}')

dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])
dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])
dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])
dict_keys(['content', 'filename'])
dict_keys(['name', 'description', 'content', 'filename'])


openclaw-main/docs/cli/status.md
openclaw-main/docs/cli/logs.md
openclaw-main/docs/cli/directory.md
openclaw-main/vendor/a2ui/README.md
openclaw-main/skills/healthcheck/SKILL.md


In [17]:
#print(f'{text_search_results[4]['content']}')
print(len(openclaw_chunks))

5681


In [18]:
## Vector search
from tqdm.auto import tqdm
import numpy as np

openclaw_embeddings = []

for d in tqdm(openclaw_chunks):
    v = embedding_model.encode(d['chunk'])
    openclaw_embeddings.append(v)


  0%|          | 0/5681 [00:00<?, ?it/s]

In [19]:
openclaw_embeddings = np.array(openclaw_embeddings)

In [20]:
from minsearch import VectorSearch
openclaw_vindex = VectorSearch()
openclaw_vindex.fit(openclaw_embeddings, openclaw_chunks)

In [30]:
query = "How can I install OpenClaw"
#query = "Give me an overview of OpenClaw"
vector_search_results = vector_search(query)
print(vector_search_results[1].keys())

dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])


In [42]:
print(vector_search_results[1].keys())

dict_keys(['start', 'chunk', 'filename'])


In [31]:
for r in vector_search_results:
    print(f"{r.keys()}")
print(f"\n")

for r in vector_search_results:
    print(f'{r['filename']}')
print(f"\n")

#for r in vector_search_results:
#    print(f'{r['chunk']}')

dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])
dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])
dict_keys(['start', 'chunk', 'filename'])
dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])
dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])


openclaw-main/docs/platforms/mac/dev-setup.md
openclaw-main/docs/install/installer.md
openclaw-main/README.md
openclaw-main/docs/install/installer.md
openclaw-main/docs/help/faq.md




In [76]:
query = "What is OpenClaw"
#query = "Give me an overview of OpenClaw"
hybrid_search_results = hybrid_search(query)

In [77]:
for r in hybrid_search_results:
    print(f"{r.keys()}")
print(f"\n")

for r in hybrid_search_results:
    print(f'{r['filename']}')
print(f"\n")

#for r in hybrid_search_results:
#    print(f'{r['chunk']}')

dict_keys(['title', 'summary', 'read_when', 'content', 'filename'])
dict_keys(['summary', 'title', 'x-i18n', 'content', 'filename'])
dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])
dict_keys(['summary', 'read_when', 'title', 'content', 'filename'])
dict_keys(['role', 'summary', 'see-also', 'content', 'filename'])
dict_keys(['start', 'chunk', 'filename'])
dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])
dict_keys(['start', 'chunk', 'summary', 'read_when', 'title', 'filename'])
dict_keys(['start', 'chunk', 'filename'])
dict_keys(['start', 'chunk', 'filename'])


openclaw-main/docs/reference/templates/USER.md
openclaw-main/docs/zh-CN/help/faq.md
openclaw-main/docs/gateway/troubleshooting.md
openclaw-main/docs/help/troubleshooting.md
openclaw-main/extensions/open-prose/skills/prose/primitives/session.md
openclaw-main/SECURITY.md
openclaw-main/docs/help/environment.md
openclaw-main/docs/plugins/community.md
openclaw-main/VISION.md
openclaw-main/

In [32]:
system_prompt = """
You are a helpful assistant for a course. 

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
"""

In [40]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    tools=[text_search],
    #tools=[vector_search],
    model='openai:gpt-4o-mini'
#    model='gpt-4o-mini'
)

In [41]:
question = "Give me an overview of OpenClaw"

result_textsearch = await agent.run(user_prompt=question)
result_textsearch

AgentRunResult(output="OpenClaw is a multi-channel communication framework designed to facilitate interactions across various messaging platforms like WhatsApp, Telegram, Discord, and others. Here’s an overview of its key components and functionalities:\n\n1. **Network Architecture**: OpenClaw uses a gateway architecture for connecting devices and managing communications between different channels and services. It supports various pairing methods to establish secure connections.\n\n2. **Installation and Logging**:\n   - OpenClaw can be installed using different methods, such as git or npm.\n   - It provides a logging framework that logs events and errors into structured files, available for monitoring through the CLI and a web-based control UI.\n\n3. **Command-Line Interface (CLI)**: \n   - OpenClaw features a command-line interface with commands for checking status, viewing logs, troubleshooting, and managing configurations. For instance, `openclaw status` provides diagnostics on the 

In [45]:
from pydantic_ai import Agent

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    #tools=[text_search],
    tools=[vector_search],
    model='openai:gpt-4o-mini'
#    model='gpt-4o-mini'
)

In [47]:
question = "Give me an overview of OpenClaw"

result_vsearch = await agent.run(user_prompt=question)
result_vsearch 

AgentRunResult(output='OpenClaw is a tool designed to serve as a personal assistant and coordination layer for various tasks, enabling automation and better organization across platforms. \n\n### Key Features:\n1. **Multi-Platform Access**: OpenClaw supports communication across numerous platforms such as WhatsApp, Telegram, and WebChat, allowing users wide access and interaction from different devices.\n  \n2. **Persistent Memory**: It retains memory and workspace across sessions, enhancing the user experience by allowing for continuity in tasks and interactions.\n  \n3. **Tool Orchestration**: OpenClaw can work with various tools, including browsers and scheduling applications. This means users can automate tasks like data collection or managing reminders seamlessly.\n  \n4. **Environment Configuration**: Users can configure OpenClaw via environment variables to customize its behavior and file structure significantly.\n\n5. **Community Plugins**: OpenClaw has a thriving community tha

In [50]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIChatModel(
    model_name="mlx-community/Mistral-7B-Instruct-v0.3-4bit",
#    model_name="mlx-community/Llama-3.2-3B-Instruct-4bit",
    provider=OpenAIProvider(
        base_url="http://localhost:8080/v1",
        api_key="not-needed",   # or just "dummy"
    ),
)

agent = Agent(
    name="faq_agent",
    instructions=system_prompt,
    #tools=[text_search],
    tools=[vector_search],
    model=model,
)

In [51]:
question = "Give me an overview of OpenClaw"
result_mlx = await agent.run(user_prompt=question)
result_mlx

AgentRunResult(output="OpenClaw is a semantic search engine that allows for natural language queries and returns a list of up to 5 search results based on the context and meaning of the query, rather than just matching keywords. It's designed to provide more accurate and relevant results by understanding the intent and semantics of the user's query. The `vector_search` function in the provided code snippet is an example of how to use OpenClaw to perform a search.")

In [52]:
result_mlx.new_messages()

[ModelRequest(parts=[UserPromptPart(content='Give me an overview of OpenClaw', timestamp=datetime.datetime(2026, 3, 29, 3, 31, 43, 100862, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 3, 29, 3, 31, 43, 101214, tzinfo=datetime.timezone.utc), instructions="You are a helpful assistant for a course. \n\nUse the search tool to find relevant information from the course materials before answering questions.\n\nIf you can find specific information through search, use it to provide accurate answers.\nIf the search doesn't return relevant results, let the user know and provide general guidance.", run_id='3bcf4e96-dff9-486d-8373-98065b3ea0ce'),
 ModelResponse(parts=[TextPart(content="OpenClaw is a semantic search engine that allows for natural language queries and returns a list of up to 5 search results based on the context and meaning of the query, rather than just matching keywords. It's designed to provide more accurate and relevant results by understanding the intent an